In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split 
from sklearn.metrics import precision_recall_curve, f1_score
from toad.metrics import KS, AUC 
from toad.plot import bin_plot, badrate_plot
import xgboost as xgb 
import pandas as pd
import seaborn as sns 
import numpy as np
import matplotlib.pyplot as plt
import toad
import pickle

## Load Data and Transform

In [ ]:
df = pd.read_excel('MB_Prediction_20251231.xlsx')

In [ ]:
df.head(5)
df.shape

In [ ]:
df.head()

In [ ]:
df_orig = df.copy(deep=True)

In [ ]:
import pandas as pd

# Assuming df is your DataFrame
unique_values = df['PROVINCE'].unique()
print(unique_values)

In [ ]:
df = df.drop(columns=['CUSTOMER_ID', 'PROVINCE', 'DISTRICT', 'OCCUP_TYPE_DESC','MB_INT_TRF_AMOUNT_USD','AVG_BALANCE_GROUP'])


In [ ]:
numeric_cols = [
    'AGE',
    'MB_INT_TRF_CNT',
    'MB_LOCAL_TRF_CNT',
    'MB_LOCAL_TRF_AMOUNT_USD',
    'MB_PAYMNENT_CNT',
    'MB_PAYMNENT_CNT.1',
    'AVG_BALANCE_USD'
]

categorical_cols = [
    'GENDER',
    'MARITAL_STATUS',
    'PROVINCE',
    'OCCUP_DET_DESC',
    
]


In [ ]:
df[numeric_cols] = df[numeric_cols].fillna(0)


In [ ]:
from sklearn.preprocessing import QuantileTransformer

df['AVG_BALANCE_USD'] = df['AVG_BALANCE_USD'].clip(lower=0)

lower_cap = df['AVG_BALANCE_USD'].quantile(0.01)
upper_cap = df['AVG_BALANCE_USD'].quantile(0.99)
df['AVG_BALANCE_USD_CAPPED'] = df['AVG_BALANCE_USD'].clip(lower=lower_cap, upper=upper_cap)


df['AVG_BALANCE_USD_LOG'] = np.log1p(df['AVG_BALANCE_USD'])
df = df.drop(columns=['AVG_BALANCE_USD','AVG_BALANCE_USD_CAPPED'])

In [ ]:
df['AVG_BALANCE_USD_LOG'].describe()

In [ ]:
df.head()

## Load ScoreCard model

In [ ]:
import pickle

In [ ]:
with open("scorecard.pkl", "rb") as f:
    load_card = pickle.load(f)

In [ ]:
df_orig['Score'] = load_card.predict(df)

In [ ]:
def reverse_score(old_sc, var_sc):
    return var_sc-old_sc

In [ ]:
def get_loan_level(df_orig, target_score='Score', out_col='Level'):
    bins = [400, 650, 750, 850, 1000]  
    labels = [
        'Poor',         
        'Fair',       
        'Good',          
        'Excellent',       
    ]
    min_sc = df_orig[target_score].min()
    max_sc = df_orig[target_score].max()
    df_orig['new_score'] = df_orig.apply(lambda x:reverse_score(x['Score'],max_sc+min_sc),axis=1)
    df_orig[out_col] = pd.cut(df_orig['new_score'], bins=bins, labels=labels, right=True)  # Categorize scores
    
    return df_orig
    
df_orig = get_loan_level(df_orig)

In [ ]:
df_orig.to_csv('Prediction20251231.csv', index = False)